# C: External Data Augmentation

This notebook augments the DC charger records produced by member A. The workflow is: validate the input, test the API on a pilot sample, review candidate matches, process the full dataset, calculate coverage, and create the handoff file.

The pilot uses 30 records to validate the API fields. Full matching, review, coverage calculation, and export then use the cached Australia-wide response.


## Objectives, Input, and Output

Read member A's `data/processed/chargers_clean.csv`, retain `record_id` as the stable join key, and filter to `charger_type_standardized == 'DC'`.

The final output is `data/processed/charger_attributes.csv`. Coverage is calculated over unique DC locations identified by latitude and longitude rounded to six decimal places. The output retains `record_id` for stable downstream database joins.


In [41]:
from pathlib import Path
import csv

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        expected = candidate / 'data' / 'processed' / 'chargers_clean.csv'
        if expected.exists():
            return candidate
    raise FileNotFoundError('Cannot find data/processed/chargers_clean.csv from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'chargers_clean.csv'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

with INPUT_FILE.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    source_rows = list(reader)
    input_columns = set(reader.fieldnames or [])

required_columns = {
    'record_id', 'Station_address', 'operator_standardized',
    'Latitude', 'Longitude', 'charger_type_standardized'
}
missing_columns = required_columns - input_columns
if missing_columns:
    raise ValueError(f'Missing required input columns: {sorted(missing_columns)}')

dc_rows = [row for row in source_rows if row['charger_type_standardized'] == 'DC']
dc_record_ids = [row['record_id'] for row in dc_rows]

if not dc_rows:
    raise ValueError('No DC records were found in the cleaned input.')
if any(not record_id for record_id in dc_record_ids):
    raise ValueError('At least one DC record has no record_id.')
if len(dc_record_ids) != len(set(dc_record_ids)):
    raise ValueError('record_id is not unique among DC records.')
if any(not row['Latitude'] or not row['Longitude'] for row in dc_rows):
    raise ValueError('At least one DC record is missing coordinates.')

print('Project root:', PROJECT_ROOT)
print('All cleaned records:', len(source_rows))
print('DC source records:', len(dc_rows))
print('Unique DC record IDs:', len(set(dc_record_ids)))


Project root: C:\Users\Administrator\Desktop\5339
All cleaned records: 1958
DC source records: 433
Unique DC record IDs: 433


## Source and Configuration

The external source is the Open Charge Map API. The API key is supplied through the `OCM_API_KEY` environment variable and is never stored in the notebook or Git.

The pilot retrieves nearby candidates for 30 DC source records. Once the Australia-wide cache exists, later runs do not require an API key. Distance is used to generate candidates but is not sufficient by itself to establish a match.


In [42]:
import getpass
import os

OCM_API_URL = 'https://api.openchargemap.io/v3/poi/'
OCM_API_KEY = os.getenv('OCM_API_KEY') or ''
cached_api_data_available = (
    (EXTERNAL_DIR / 'ocm_pilot_raw.json').exists()
    and (EXTERNAL_DIR / 'ocm_australia_raw.json').exists()
)
if not OCM_API_KEY and not cached_api_data_available:
    OCM_API_KEY = getpass.getpass('Paste Open Charge Map API key: ')

PILOT_SIZE = min(30, len(dc_rows))
SEARCH_RADIUS_KM = 1.0
MAX_CANDIDATES_PER_RECORD = 10

print('Pilot records:', PILOT_SIZE)
print('Candidate search radius (km):', SEARCH_RADIUS_KM)
print('OCM_API_KEY configured:', bool(OCM_API_KEY))
if not OCM_API_KEY and not cached_api_data_available:
    raise RuntimeError('Open Charge Map API key was not provided and no local cache is available.')
if not OCM_API_KEY and cached_api_data_available:
    print('Using local OCM caches; an API key is not required for this run.')


Pilot records: 30
Candidate search radius (km): 1.0
OCM_API_KEY configured: True


## Retrieval and Local Cache

Query nearby candidates for each pilot record and save the complete response to `data/external/ocm_pilot_raw.json`. Existing cache files are reused by default to avoid repeated requests and unnecessary API usage.


In [43]:
import json
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import Request, urlopen

PILOT_CACHE = EXTERNAL_DIR / 'ocm_pilot_raw.json'

def retrieve_ocm_candidates(row: dict, api_key: str) -> list:
    params = {
        'output': 'json',
        'latitude': row['Latitude'],
        'longitude': row['Longitude'],
        'distance': SEARCH_RADIUS_KM,
        'distanceunit': 'km',
        'maxresults': MAX_CANDIDATES_PER_RECORD,
        'compact': 'false',
        'verbose': 'false',
    }
    request = Request(
        f'{OCM_API_URL}?{urlencode(params)}',
        headers={
            'X-API-Key': api_key,
            'User-Agent': 'COMP5339-Assignment1-Haihui/1.0',
        },
    )
    with urlopen(request, timeout=60) as response:
        return json.load(response)

if PILOT_CACHE.exists():
    with PILOT_CACHE.open(encoding='utf-8') as handle:
        pilot_payload = json.load(handle)
    print('Loaded cached pilot response:', PILOT_CACHE)
elif OCM_API_KEY:
    pilot_payload = {
        'source': 'Open Charge Map API v3',
        'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
        'search_radius_km': SEARCH_RADIUS_KM,
        'records': [],
    }
    for index, row in enumerate(dc_rows[:PILOT_SIZE], start=1):
        candidates = retrieve_ocm_candidates(row, OCM_API_KEY)
        pilot_payload['records'].append({
            'record_id': row['record_id'],
            'input_address': row['Station_address'],
            'input_operator': row['operator_standardized'],
            'input_latitude': float(row['Latitude']),
            'input_longitude': float(row['Longitude']),
            'candidates': candidates,
        })
        print(f'Retrieved {index}/{PILOT_SIZE}: {len(candidates)} candidates')
    with PILOT_CACHE.open('w', encoding='utf-8') as handle:
        json.dump(pilot_payload, handle, ensure_ascii=False, indent=2)
    print('Saved pilot cache:', PILOT_CACHE)
else:
    pilot_payload = None
    print('Retrieval skipped because OCM_API_KEY is not configured and no cache exists.')


Loaded cached pilot response: C:\Users\Administrator\Desktop\5339\data\external\ocm_pilot_raw.json


## Station Matching

Latitude and longitude generate candidate sites. Operator, postcode, address-token overlap, compatible power, and distance contribute to the match score. Each source record retains at most one highest-ranked candidate supported by multiple pieces of evidence. Reproducible review overrides reject clear address conflicts and accept a small number of network-name changes with sufficient location evidence.


In [44]:
import math

def haversine_metres(lat1, lon1, lat2, lon2):
    radius_m = 6_371_008.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    value = (
        math.sin(delta_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    )
    return 2 * radius_m * math.asin(math.sqrt(value))

def connection_titles(candidate):
    titles = []
    for connection in candidate.get('Connections') or []:
        title = (connection.get('ConnectionType') or {}).get('Title')
        if title and title not in titles:
            titles.append(title)
    return ' | '.join(titles)

def connection_power_values(candidate):
    values = {
        float(connection['PowerKW'])
        for connection in candidate.get('Connections') or []
        if connection.get('PowerKW') is not None
    }
    return ' | '.join(f'{value:g}' for value in sorted(values))

def connection_quantity_total(candidate):
    quantities = [
        connection.get('Quantity')
        for connection in candidate.get('Connections') or []
        if connection.get('Quantity') is not None
    ]
    return sum(quantities) if quantities else None

# Pilot decisions are evidence-based review results, not an automatic matching rule.
PILOT_REVIEW_SEED = {
    '272521': ('accept', 'Same operator and postcode; Wallgrove Road location; 65.1 m away.'),
    '170824': ('reject', 'Input is Wilcannia 2836 but candidate is Armidale 2350.'),
    '460615': ('reject', 'Wrong operator and town; candidate is a Tesla site in Armidale.'),
    '274592': ('reject', 'Wrong operator, charger type and town; candidate is an Armidale motel.'),
    '311308': ('reject', 'Destination charger is 22 kW; another Tesla candidate better matches the 130 kW source record.'),
    '100404': ('accept', 'Same Bay Street location and Tesla operator; Supercharger power best matches the source record.'),
    '272613': ('reject', 'Operator, address and power do not match the source record.'),
    '272632': ('reject', 'Address and operator do not match the source record.'),
    '272688': ('accept', 'Same operator, street address, postcode and 50 kW power.'),
    '192929': ('accept', 'Same JOLT operator and exact 1 Park Street address; 5.7 m away.'),
    '190757': ('reject', 'Another JOLT candidate has the exact source address and is closer.'),
    '157566': ('reject', 'Operator and address do not match the Evie source record.'),
    '190706': ('manual_review', 'Same town and operator, but address and power differ; verify on a map before acceptance.'),
    '209950': ('reject', 'Operator and address do not match the Evie source record.'),
    '121552': ('reject', 'Wrong operator and address for the Evie Bathurst source record.'),
    '191953': ('reject', 'Wrong operator and address for the Evie Bathurst source record.'),
    '79311': ('accept', 'Same Tesla operator, postcode and coordinates; power is consistent with a Supercharger.'),
    '172921': ('accept', 'Same Chargefox operator, Oxley Highway address and postcode; 29.9 m away.'),
    '480398': ('reject', 'Wrong operator and weaker address match than the accepted Chargefox candidate.'),
    '172920': ('reject', 'Wrong operator and weaker address match than the accepted Chargefox candidate.'),
    '190682': ('reject', 'Input is Walcha 2354 but candidate is Brewarrina 2839.'),
    '190394': ('accept', 'Same NRMA operator and Ewingsdale address; 48.0 m away.'),
    '190875': ('accept', 'Same JOLT operator, exact address, postcode and 25 kW power.'),
    '150024': ('reject', 'Wrong operator, connector and power; nearly 1 km away.'),
    '191062': ('reject', 'Wrong operator, address, postcode and power; nearly 1 km away.'),
    '295979': ('accept', 'Same JOLT operator, Barrenjoey Road location and 25 kW power; 23.5 m away.'),
    '278525': ('reject', 'Wrong operator and planned status; candidate does not represent the JOLT source record.'),
    '190677': ('reject', 'Input is Scone 2337 but candidate is Broken Hill 2880.'),
    '272624': ('accept', 'Same operator, exact Horsley Drive address, postcode and two connections.'),
}

CANDIDATE_REVIEW_FILE = EXTERNAL_DIR / 'ocm_pilot_candidate_review.csv'
existing_review = {}
if CANDIDATE_REVIEW_FILE.exists():
    with CANDIDATE_REVIEW_FILE.open(encoding='utf-8-sig', newline='') as handle:
        for reviewed_row in csv.DictReader(handle):
            key = (reviewed_row['record_id'], reviewed_row['external_station_id'])
            decision = reviewed_row.get('review_decision', '').strip()
            notes = reviewed_row.get('review_notes', '').strip()
            if decision or notes:
                existing_review[key] = (decision, notes)

dc_by_id = {row['record_id']: row for row in dc_rows}
candidate_rows = []
if pilot_payload:
    for source in pilot_payload['records']:
        source_record = dc_by_id[source['record_id']]
        for candidate in source['candidates']:
            address = candidate.get('AddressInfo') or {}
            operator = candidate.get('OperatorInfo') or {}
            status = candidate.get('StatusType') or {}
            usage = candidate.get('UsageType') or {}
            candidate_lat = address.get('Latitude')
            candidate_lon = address.get('Longitude')
            distance_m = None
            if candidate_lat is not None and candidate_lon is not None:
                distance_m = haversine_metres(
                    source['input_latitude'], source['input_longitude'],
                    float(candidate_lat), float(candidate_lon),
                )
            review_key = (source['record_id'], str(candidate.get('ID')))
            review_decision, review_notes = existing_review.get(
                review_key,
                PILOT_REVIEW_SEED.get(str(candidate.get('ID')), ('', '')),
            )
            candidate_rows.append({
                'record_id': source['record_id'],
                'external_station_id': candidate.get('ID'),
                'distance_m': None if distance_m is None else round(distance_m, 1),
                'input_operator': source['input_operator'],
                'external_operator': operator.get('Title'),
                'input_address': source['input_address'],
                'input_power_kw': source_record.get('power_kw'),
                'input_plug_count': source_record.get('Number_of_plugs'),
                'external_title': address.get('Title'),
                'external_address_line_1': address.get('AddressLine1'),
                'external_address_line_2': address.get('AddressLine2'),
                'external_town': address.get('Town'),
                'external_postcode': address.get('Postcode'),
                'connector_types': connection_titles(candidate),
                'external_power_kw_values': connection_power_values(candidate),
                'external_connection_quantity': connection_quantity_total(candidate),
                'external_number_of_points': candidate.get('NumberOfPoints'),
                'external_status': status.get('Title'),
                'external_usage_type': usage.get('Title'),
                'external_usage_cost': candidate.get('UsageCost'),
                'external_last_verified': candidate.get('DateLastVerified'),
                'review_decision': review_decision,
                'review_notes': review_notes,
            })

if candidate_rows:
    with CANDIDATE_REVIEW_FILE.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(candidate_rows[0]))
        writer.writeheader()
        writer.writerows(candidate_rows)
    print('Candidate rows:', len(candidate_rows))
    print('Review file:', CANDIDATE_REVIEW_FILE)
    print('Fill review_decision with accept or reject after checking distance, address, and operator.')
else:
    print('No candidate rows are available yet.')


Candidate rows: 31
Review file: C:\Users\Administrator\Desktop\5339\data\external\ocm_pilot_candidate_review.csv
Fill review_decision with accept or reject after checking distance, address, and operator.


## Full External Retrieval and Local Matching

After validating the fields and matching risks with the pilot, retrieve Australian Open Charge Map POIs in pages and cache the response locally. Match the 433 DC source records against this cache. Sites within 3 km become spatial candidates; strong operator, postcode, and address evidence can also recover records whose source coordinates are clearly inconsistent with their addresses.

Candidates are ranked using weighted address, operator, postcode, power, and distance evidence. Automatic acceptance requires the highest-ranked candidate to satisfy multiple identity checks. A reviewed `review_decision` overrides the automatic suggestion. Because Open Charge Map aggregates contributions from multiple providers, the output retains the data provider and last verification date.


In [45]:
FULL_CACHE = EXTERNAL_DIR / 'ocm_australia_raw.json'
FULL_REVIEW_FILE = EXTERNAL_DIR / 'ocm_full_candidate_review.csv'
from urllib.error import HTTPError
import time

FULL_PAGE_SIZE = 1_000

if FULL_CACHE.exists():
    with FULL_CACHE.open(encoding='utf-8') as handle:
        full_payload = json.load(handle)
    print('Loaded cached Australia response:', FULL_CACHE)
elif OCM_API_KEY:
    australia_sites = []
    greater_than_id = 0
    page_number = 0
    while True:
        full_params = {
            'output': 'json',
            'countrycode': 'AU',
            'maxresults': FULL_PAGE_SIZE,
            'sortby': 'id_asc',
            'greaterthanid': greater_than_id,
            'compact': 'false',
            'verbose': 'false',
        }
        full_request = Request(
            f'{OCM_API_URL}?{urlencode(full_params)}',
            headers={
                'X-API-Key': OCM_API_KEY,
                'User-Agent': 'COMP5339-Assignment1-Haihui/1.0',
            },
        )
        try:
            with urlopen(full_request, timeout=180) as response:
                page = json.load(response)
        except HTTPError as error:
            error_body = error.read().decode('utf-8', errors='replace')
            raise RuntimeError(f'Open Charge Map HTTP {error.code}: {error_body}') from error
        if not page:
            break
        australia_sites.extend(page)
        page_number += 1
        greater_than_id = max(int(site['ID']) for site in page)
        print(f'Downloaded Australia page {page_number}: {len(page)} records; total {len(australia_sites)}')
        if len(page) < FULL_PAGE_SIZE:
            break
        time.sleep(0.25)
    full_payload = {
        'source': 'Open Charge Map API v3',
        'country_code': 'AU',
        'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
        'license_note': 'Retain the DataProvider attribution supplied for each POI.',
        'records': australia_sites,
    }
    with FULL_CACHE.open('w', encoding='utf-8') as handle:
        json.dump(full_payload, handle, ensure_ascii=False, indent=2)
    print('Saved Australia cache:', FULL_CACHE)
else:
    full_payload = None
    print('Full retrieval skipped: set OCM_API_KEY or provide the cached Australia response.')

if full_payload:
    print('Australia OCM locations:', len(full_payload['records']))


HTTPError: HTTP Error 400: Bad Request

In [ ]:
import re

FULL_SEARCH_RADIUS_KM = 3.0

OPERATOR_ALIASES = {
    'tesla': ('tesla',),
    'jolt': ('jolt',),
    'evie': ('evie',),
    'nrma': ('nrma',),
    'chargefox': ('chargefox',),
    'bp': ('bp', 'bp pulse'),
    'ampol': ('ampol', 'ampcharge'),
    'exploren': ('exploren',),
    'engie': ('engie',),
    'everty': ('everty',),
    'wevolt': ('wevolt',),
}
ADDRESS_STOP_WORDS = {
    'australia', 'nsw', 'new', 'south', 'wales', 'sydney',
    'road', 'rd', 'street', 'st', 'highway', 'hwy', 'drive', 'dr',
    'avenue', 'ave', 'charger', 'charging', 'station', 'centre', 'center',
    'the', 'at', 'and',
}

def canonical_operator(value):
    text = re.sub(r'[^a-z0-9]+', ' ', (value or '').casefold()).strip()
    for canonical, aliases in OPERATOR_ALIASES.items():
        if any(re.search(rf'\b{re.escape(alias)}\b', text) for alias in aliases):
            return canonical
    return text

def address_tokens(*values):
    text = ' '.join(value or '' for value in values).casefold()
    tokens = set(re.findall(r'[a-z0-9]+', text))
    return {
        token for token in tokens
        if token not in ADDRESS_STOP_WORDS and not (token.isdigit() and len(token) == 4)
    }

def source_postcode(row):
    postcode = (row.get('postcode_standardized') or '').strip()
    if postcode:
        return postcode
    matches = re.findall(r'(?<!\d)(\d{4})(?!\d)', row.get('Station_address') or '')
    return matches[-1] if matches else ''

def candidate_power_values(candidate):
    return [
        float(connection['PowerKW'])
        for connection in candidate.get('Connections') or []
        if connection.get('PowerKW') is not None
    ]

def power_is_compatible(source_power, candidate):
    if source_power in (None, ''):
        return False
    source_value = float(source_power)
    return any(abs(value - source_value) / max(source_value, 1.0) <= 0.40 for value in candidate_power_values(candidate))

# Evidence-based overrides for borderline rows found during full-file review.
# These decisions prevent known address conflicts from being counted only to reach the target.
FULL_REVIEW_SEED = {
    ('ev_13956b6342d4b2caac28de8cd39e00ab65299a93f966f7fa6f712f97175c062c', '273195'): ('reject', 'Different Byron Bay street address despite matching operator and postcode.'),
    ('ev_b7595fee109180975840a8cfa34b5d289b18baf0842f3347fe1dd4123652fe52', '273195'): ('reject', 'Different Byron Bay street address despite matching operator and postcode.'),
    ('ev_03205191e7e39cc26d8b252b733ae8e6f3e6cb77ee9beaf481df58bb3677123c', '190875'): ('reject', 'Different Anzac Parade street number and postcode; 4.3 km separation.'),
    ('ev_4d6b8dff7317e9dafe5b708263d0986eee6bb518afd0ea10c1156623d90c3523', '297042'): ('reject', 'Hazelbrook and Glenbrook addresses differ; 16.4 km separation.'),
    ('ev_d78ba96b61211f7f07c87c28f8d301944e29e3abccfb867c8d0a8e1389610076', '271003'): ('reject', 'Hornsby source does not match the Kerang OCM site.'),
    ('ev_c9f5c210274f61224f4cf5ed9c3ea07a499e7f97fcead507898565efe3ce2942', '170607'): ('reject', 'Different Seven Hills street address; 2.2 km separation.'),
    ('ev_ac4323b62f6de476dee033a103a04c7361381acdc02c4f59ae8df5820084575c', '131610'): ('reject', 'Main campus source and Innovation Campus candidate are separate sites.'),
    ('ev_3a46df09a2d19b8dd5c0d3aeaf5466a14559c8ea01a349b2112c326b6a27a245', '190003'): ('accept', 'Coordinates are 8.5 m apart and postcode and Sydney Road location agree.'),
    ('ev_59f949f85f20276de5b579d641f6575556cf987dafea00517f88d2e261a02284', '274452'): ('accept', 'Same Gunnedah postcode, compatible power and nearby town-centre carpark.'),
    ('ev_874f423dd1a3f49848fbde04fc442b77c4de522dd5238f905a2e0af8c5c94bd2', '272632'): ('accept', 'Same Evie operator, postcode and power; Pole Lane source is 419 m from the Christie Street OCM site.'),
}
existing_full_review = dict(FULL_REVIEW_SEED)
if FULL_REVIEW_FILE.exists():
    with FULL_REVIEW_FILE.open(encoding='utf-8-sig', newline='') as handle:
        for reviewed_row in csv.DictReader(handle):
            key = (reviewed_row['record_id'], reviewed_row['external_station_id'])
            decision = reviewed_row.get('review_decision', '').strip()
            notes = reviewed_row.get('review_notes', '').strip()
            if decision or notes:
                existing_full_review[key] = (decision, notes)

full_candidate_rows = []
source_records_with_candidates = set()
if full_payload:
    ocm_sites = full_payload['records']
    for source in dc_rows:
        source_lat = float(source['Latitude'])
        source_lon = float(source['Longitude'])
        source_tokens = address_tokens(source['Station_address'], source.get('Station_name'))
        source_operator = canonical_operator(source['operator_standardized'])
        source_pc = source_postcode(source)
        ranked = []
        for candidate in ocm_sites:
            address = candidate.get('AddressInfo') or {}
            candidate_lat = address.get('Latitude')
            candidate_lon = address.get('Longitude')
            if candidate_lat is None or candidate_lon is None:
                continue
            distance_m = haversine_metres(source_lat, source_lon, float(candidate_lat), float(candidate_lon))
            operator = candidate.get('OperatorInfo') or {}
            candidate_operator = canonical_operator(operator.get('Title'))
            candidate_pc = str(address.get('Postcode') or '').strip()
            candidate_tokens = address_tokens(
                address.get('Title'), address.get('AddressLine1'),
                address.get('AddressLine2'), address.get('Town'),
            )
            shared_token_count = len(source_tokens & candidate_tokens)
            overlap = shared_token_count / max(1, min(len(source_tokens), len(candidate_tokens)))
            operator_match = bool(source_operator and source_operator == candidate_operator)
            postcode_match = bool(source_pc and candidate_pc and source_pc == candidate_pc)
            power_match = power_is_compatible(source.get('power_kw'), candidate)
            address_match = overlap >= 0.25
            include_candidate = (
                distance_m <= FULL_SEARCH_RADIUS_KM * 1000
                or (postcode_match and (operator_match or address_match))
                or (operator_match and overlap >= 0.50)
            )
            if not include_candidate:
                continue
            score = (
                4 * operator_match + 3 * postcode_match
                + 6 * (overlap >= 0.75)
                + 4 * (0.50 <= overlap < 0.75)
                + 2 * (0.25 <= overlap < 0.50)
                + 1 * power_match
                + 4 * (distance_m <= 75)
                + 2 * (75 < distance_m <= 250)
                + 1 * (250 < distance_m <= 750)
            )
            ranked.append((score, distance_m, overlap, shared_token_count, operator_match, postcode_match, power_match, candidate))
        ranked.sort(key=lambda item: (-item[0], item[1]))
        if ranked:
            source_records_with_candidates.add(source['record_id'])
        top_score_margin = (
            ranked[0][0] - ranked[1][0] if len(ranked) > 1 else ranked[0][0]
        ) if ranked else 0
        for rank, item in enumerate(ranked, start=1):
            score, distance_m, overlap, shared_token_count, operator_match, postcode_match, power_match, candidate = item
            address = candidate.get('AddressInfo') or {}
            operator = candidate.get('OperatorInfo') or {}
            status = candidate.get('StatusType') or {}
            usage = candidate.get('UsageType') or {}
            provider = candidate.get('DataProvider') or {}
            address_match = overlap >= 0.25
            strong_address_match = overlap >= 0.50 and shared_token_count >= 2
            support_count = sum((operator_match, postcode_match, address_match, power_match))
            high_confidence = (
                (distance_m <= 100 and support_count >= 2)
                or (distance_m <= 250 and support_count >= 2 and (operator_match or (postcode_match and address_match)))
                or (distance_m <= 1000 and (
                    (operator_match and postcode_match)
                    or (postcode_match and address_match)
                    or (operator_match and address_match)
                ))
                or (operator_match and postcode_match and overlap >= 0.50)
                or (distance_m <= 250 and strong_address_match and power_match)
                or (operator_match and strong_address_match and power_match)
            )
            tie_safe = distance_m <= 75 and operator_match and overlap >= 0.50 and power_match
            if rank == 1 and high_confidence and (top_score_margin >= 1 or tie_safe):
                suggested = 'auto_accept'
                if operator_match and postcode_match and overlap >= 0.50:
                    match_method = 'operator_postcode_address'
                elif operator_match and strong_address_match and power_match:
                    match_method = 'operator_address_power'
                elif distance_m <= 250 and strong_address_match and power_match:
                    match_method = 'close_address_power'
                elif distance_m <= 100:
                    match_method = 'close_coordinates_plus_identity'
                else:
                    match_method = 'combined_evidence'
                reason = f'High-confidence top candidate selected by {match_method}.'
            elif rank == 1 and score >= 5:
                match_method = 'manual_review'
                suggested = 'manual_review'
                reason = 'Top candidate has partial evidence but does not satisfy the high-confidence rule.'
            else:
                match_method = 'rejected_candidate'
                suggested = 'reject'
                reason = 'Not the strongest candidate or evidence is insufficient.'
            review_key = (source['record_id'], str(candidate.get('ID')))
            review_decision, review_notes = existing_full_review.get(review_key, ('', ''))
            full_candidate_rows.append({
                'record_id': source['record_id'],
                'external_station_id': candidate.get('ID'),
                'candidate_rank': rank,
                'match_score': score,
                'score_margin': top_score_margin if rank == 1 else '',
                'match_method': match_method,
                'distance_m': round(distance_m, 1),
                'operator_match': operator_match,
                'postcode_match': postcode_match,
                'address_token_overlap': round(overlap, 3),
                'shared_address_token_count': shared_token_count,
                'power_compatible': power_match,
                'input_operator': source['operator_standardized'],
                'external_operator': operator.get('Title'),
                'input_address': source['Station_address'],
                'input_power_kw': source.get('power_kw'),
                'input_plug_count': source.get('Number_of_plugs'),
                'external_title': address.get('Title'),
                'external_address_line_1': address.get('AddressLine1'),
                'external_address_line_2': address.get('AddressLine2'),
                'external_town': address.get('Town'),
                'external_postcode': address.get('Postcode'),
                'connector_types': connection_titles(candidate),
                'external_power_kw_values': connection_power_values(candidate),
                'external_connection_quantity': connection_quantity_total(candidate),
                'external_number_of_points': candidate.get('NumberOfPoints'),
                'external_status': status.get('Title'),
                'external_usage_type': usage.get('Title'),
                'external_usage_cost': candidate.get('UsageCost'),
                'external_last_verified': candidate.get('DateLastVerified'),
                'data_provider': provider.get('Title'),
                'suggested_decision': suggested,
                'suggested_reason': reason,
                'review_decision': review_decision,
                'review_notes': review_notes,
            })

if full_candidate_rows:
    with FULL_REVIEW_FILE.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(full_candidate_rows[0]))
        writer.writeheader()
        writer.writerows(full_candidate_rows)
    suggested_accept_ids = {
        row['record_id'] for row in full_candidate_rows
        if row['suggested_decision'] == 'auto_accept'
    }
    print('Full candidate rows:', len(full_candidate_rows))
    print('DC records with at least one candidate:', len(source_records_with_candidates))
    print('Strict auto-accept suggestions:', len(suggested_accept_ids))
    print(f'Strict suggested coverage: {len(suggested_accept_ids) / len(dc_rows):.2%}')
    print('Full review file:', FULL_REVIEW_FILE)
else:
    print('No full candidate review file was generated.')


## Validation and Coverage

The assignment specifies coverage of DC charger locations. The denominator therefore uses unique locations defined by latitude and longitude rounded to six decimal places: 433 DC source records represent 430 unique locations. `auto_accept` applies only to the highest-ranked candidate supported by multiple identity checks, and a reviewed `review_decision` overrides that suggestion. A match contributes to coverage only when it supplies at least one new OCM attribute.

Limitations: some TfNSW records contain conflicts between their address and coordinates. OCM is contributor-maintained, so operator, status, price, and verification dates may be outdated. Match method, distance, score, external ID, and data provider are retained to support later auditing.


In [ ]:
accepted_matches = []
if FULL_REVIEW_FILE.exists():
    with FULL_REVIEW_FILE.open(encoding='utf-8-sig', newline='') as handle:
        reviewed_candidates = list(csv.DictReader(handle))
    for row in reviewed_candidates:
        manual_decision = row.get('review_decision', '').strip().casefold()
        accepted = (
            manual_decision == 'accept'
            or (not manual_decision and row.get('suggested_decision') == 'auto_accept')
        )
        has_new_attribute = any(row.get(field, '').strip() for field in (
            'connector_types', 'external_connection_quantity',
            'external_number_of_points', 'external_status',
            'external_usage_type', 'external_usage_cost',
        ))
        if accepted and manual_decision != 'reject' and has_new_attribute:
            accepted_matches.append(row)

accepted_by_id = {}
for row in accepted_matches:
    accepted_by_id.setdefault(row['record_id'], row)
accepted_record_ids = set(accepted_by_id)
def location_key(row):
    return (round(float(row['Latitude']), 6), round(float(row['Longitude']), 6))

dc_location_keys = {location_key(row) for row in dc_rows}
accepted_location_keys = {location_key(dc_by_id[record_id]) for record_id in accepted_record_ids}
location_coverage = len(accepted_location_keys) / len(dc_location_keys)
coverage_target = math.ceil(len(dc_location_keys) * 0.5)
print('Accepted unique DC source records:', len(accepted_record_ids))
print('Accepted unique DC locations:', len(accepted_location_keys))
print('Total unique DC locations:', len(dc_location_keys))
print(f'Accepted location coverage: {location_coverage:.2%}')
print('Target at unique-location denominator:', coverage_target)
print('Coverage target met:', len(accepted_location_keys) >= coverage_target)


## Export and Handoff

The final file joins to member A's cleaned data through `record_id`, with at most one highest-ranked OCM site per source record. An OCM site may contain multiple connectors; connector types and power values are combined with ` | ` while connection quantity and point counts are retained. The file also records match distance, score, method, and decision for audit during member D's database integration.


In [ ]:
OUTPUT_FILE = PROCESSED_DIR / 'charger_attributes.csv'

output_fields = [
    'record_id', 'external_source', 'external_station_id',
    'connector_types', 'external_power_kw_values',
    'external_connection_quantity', 'external_number_of_points',
    'external_status', 'external_usage_type', 'external_usage_cost',
    'external_last_verified', 'data_provider',
    'match_distance_m', 'match_score', 'match_method', 'match_decision',
    'external_station_title', 'external_address',
]
output_rows = []
for record_id, row in accepted_by_id.items():
    manual_decision = row.get('review_decision', '').strip().casefold()
    output_rows.append({
        'record_id': record_id,
        'external_source': 'Open Charge Map API v3',
        'external_station_id': row['external_station_id'],
        'connector_types': row['connector_types'],
        'external_power_kw_values': row['external_power_kw_values'],
        'external_connection_quantity': row['external_connection_quantity'],
        'external_number_of_points': row['external_number_of_points'],
        'external_status': row['external_status'],
        'external_usage_type': row['external_usage_type'],
        'external_usage_cost': row['external_usage_cost'],
        'external_last_verified': row['external_last_verified'],
        'data_provider': row['data_provider'],
        'match_distance_m': row['distance_m'],
        'match_score': row['match_score'],
        'match_method': row['match_method'],
        'match_decision': 'manual_accept' if manual_decision == 'accept' else 'rule_based_accept',
        'external_station_title': row['external_title'],
        'external_address': ', '.join(part for part in (
            row['external_address_line_1'], row['external_address_line_2'],
            row['external_town'], row['external_postcode'],
        ) if part),
    })

if len(accepted_location_keys) < coverage_target:
    raise RuntimeError(
        f'Coverage target not met: {len(accepted_location_keys)}/{len(dc_location_keys)} accepted locations.'
    )
with OUTPUT_FILE.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=output_fields)
    writer.writeheader()
    writer.writerows(sorted(output_rows, key=lambda row: row['record_id']))
print('Saved augmented attributes:', OUTPUT_FILE)
print('Output rows:', len(output_rows))
print(f'Final DC location coverage: {len(accepted_location_keys) / len(dc_location_keys):.2%}')
